# Libraries & Data Import

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mpsynthetics/b2b-saas-churn-dataset")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(os.path.join(path, 'MP_Synthetics_50000_Rows.csv'))

In [ ]:
sns.set_style('whitegrid')

# Data Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
print(f'Missing values: {df.isna().sum().sum()}')
print(f'Duplicated values: {df.duplicated().sum()}')

In [ ]:
df['Customer_ID'].nunique()

In [ ]:
df.describe()

In [ ]:
len(df[df['Active_Seats_Used'] > df['Total_Seats_Allowed']])

# Preprocessing

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['last_active_date'] = pd.to_datetime(df['last_active_date'], errors='coerce')

In [ ]:
df['feature_adoption_rate'] = df['feature_adoption_rate'].str.rstrip('%').astype(float)

In [ ]:
df.head()

# Feature Engineering

**New features**:
* seat_utilization - percentage of occupied seats
* days_since_last_active - days between the account's last activity and the latest activity date in the data
* tenure_days - account lifespan in days: from signup to the latest activity date in the data
* tickets_per_seat - support tickets in the last 30 days per allowed seat
* mrr_per_seat - monthly recurring revenue per allowed seat
* MRR_segment - client segment by price list: Mass, Economy, Business, Premium, Enterprise

In [ ]:
df['seat_utilization'] = round((df['active_seats_used'] / df['total_seats_allowed']) * 100, 1)

max_date = df['last_active_date'].max()
df['days_since_last_active'] = (max_date - df['last_active_date']).dt.days
df['tenure_days'] = (max_date - df['signup_date']).dt.days

df['tickets_per_seat'] = df['support_tickets_30d'] / df['total_seats_allowed']
df['mrr_per_seat'] = df['monthly_recurring_revenue'] / df['total_seats_allowed']

bins = [0, 79, 129, 299, 1600, np.inf]
labels = ['Mass', 'Economy', 'Business', 'Premium', 'Enterprise']
df['MRR_segment'] = pd.cut(df['monthly_recurring_revenue'], bins=bins, labels=labels)

In [ ]:
df['monthly_recurring_revenue'].describe(percentiles=[.5, .75, .9, .95, .99])

In [ ]:
df.groupby('MRR_segment', observed=True)['monthly_recurring_revenue'].describe()

In [ ]:
mrr = df['monthly_recurring_revenue']
print(mrr[mrr > 499].sort_values().head(20))
print(mrr.between(500, 1601).sum())

In [ ]:
df['MRR_segment'].value_counts()

# Data Export

In [ ]:
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/clean_data.csv', index=False)

# EDA

In [ ]:
churned_customers = int(df['is_churned'].sum())
churned_revenue = df.loc[df['is_churned'] == 1, 'monthly_recurring_revenue'].sum()
revenue_sum = df['monthly_recurring_revenue'].sum()

churn_rate = round(churned_customers / len(df) * 100, 1)
mrr_churn = round(churned_revenue / revenue_sum * 100, 1)
print(f"Churn rate: {churn_rate}% ({churned_customers} customers)")
print(f"MRR churn rate: {mrr_churn}% ({churned_revenue})")

In [ ]:
os.makedirs('../images', exist_ok=True)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

for axis, col in zip(ax, ['plan_tier', 'billing_cycle', 'payment_status']):
    stats = df.groupby(col, observed=True)['is_churned'].agg(clients='count', churn_rate='mean')
    sns.barplot(x=stats.index.astype(str), y=stats['churn_rate'].values * 100, ax=axis, color='#4C72B0')
    axis.bar_label(axis.containers[0], labels=[f'{r * 100:.1f}%\n(n={n:,})' for r, n in zip(stats['churn_rate'], stats['clients'])], padding=3)
    axis.set_title(f'Churn Rate by {col}')
    axis.set_ylabel('Churn rate, %')
    axis.margins(y=0.2)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(21, 6))

for axis, col in zip(ax, ['industry', 'country', 'account_manager']):
    stats = df.groupby(col, observed=True)['is_churned'].agg(clients='count', churn_rate='mean').sort_values('churn_rate', ascending=False)
    sns.barplot(x=stats['churn_rate'].values * 100, y=stats.index.astype(str), orient='h', ax=axis, color='#4C72B0')
    axis.bar_label(axis.containers[0], labels=[f'{r * 100:.1f}% (n={n:,})' for r, n in zip(stats['churn_rate'], stats['clients'])], padding=3)
    axis.set_title(f'Churn Rate by {col}')
    axis.set_xlabel('Churn rate, %')
    axis.margins(x=0.3)

plt.tight_layout()
plt.show()

In [ ]:
company_size_order = ['1-10', '11-50', '51-200', '201-500', '501-1000', '1000+']
stats = df.groupby('company_size')['is_churned'].agg(clients='count', churn_rate='mean').loc[company_size_order]

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(x=stats.index, y=stats['churn_rate'].values * 100, ax=ax, color='#4C72B0')
ax.bar_label(ax.containers[0], labels=[f'{r * 100:.1f}%\n(n={n:,})' for r, n in zip(stats['churn_rate'], stats['clients'])], padding=3)
ax.set_title('Churn Rate by Company Size')
ax.set_xlabel('Company size')
ax.set_ylabel('Churn rate, %')
ax.margins(y=0.2)

plt.show()

In [ ]:
segment_stats = df.groupby('MRR_segment', observed=True).agg(
    clients=('customer_id', 'count'),
    mrr=('monthly_recurring_revenue', 'sum')
)
segment_stats['share_of_clients'] = segment_stats['clients'] / segment_stats['clients'].sum() * 100
segment_stats['share_of_mrr'] = segment_stats['mrr'] / segment_stats['mrr'].sum() * 100

segment_stats.round(1)

In [ ]:
ax = segment_stats[['share_of_clients', 'share_of_mrr']].plot(kind='bar', figsize=(9, 5), color=['#4C72B0', '#DD8452'], rot=0)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

ax.set_title('Share of Clients vs Share of MRR by MRR Segment')
ax.set_xlabel('MRR segment')
ax.set_ylabel('Share, %')
ax.legend(['Share of clients', 'Share of MRR'])

plt.tight_layout()
plt.savefig('../images/mrr_segments.png', dpi=150)
plt.show()

In [ ]:
payment_stats = df.groupby('payment_status')['is_churned'].agg(clients='count', churn_rate='mean')

adoption_bins = pd.qcut(df['feature_adoption_rate'], 5, duplicates='drop')
adoption_stats = df.groupby(adoption_bins, observed=True)['is_churned'].agg(clients='count', churn_rate='mean')
adoption_stats.index = [f'{iv.left:.0f}-{iv.right:.0f}' for iv in adoption_stats.index]

seat_bins = pd.qcut(df['seat_utilization'], 5, duplicates='drop')
seat_stats = df.groupby(seat_bins, observed=True)['is_churned'].agg(clients='count', churn_rate='mean')
seat_stats.index = [f'{iv.left:.0f}-{iv.right:.0f}' for iv in seat_stats.index]

tickets_bins = pd.cut(df['support_tickets_30d'], bins=[-1, 2, 3, 4, 5, 8], labels=['0-2', '3', '4', '5', '6-8'])
tickets_stats = df.groupby(tickets_bins, observed=True)['is_churned'].agg(clients='count', churn_rate='mean')

In [ ]:
driver_tables = [
    ('Payment Status', 'Status', payment_stats),
    ('Feature Adoption Rate (quintiles)', 'Adoption rate, %', adoption_stats),
    ('Seat Utilization (quintiles)', 'Seat utilization, %', seat_stats),
    ('Support Tickets (30 days)', 'Tickets', tickets_stats)
]

fig, ax = plt.subplots(1, 4, figsize=(20, 5.5), sharey=True)

for axis, (title, xlabel, stats) in zip(ax, driver_tables):
    sns.barplot(x=stats.index.astype(str), y=stats['churn_rate'].values * 100, ax=axis, color='#4C72B0')
    axis.bar_label(axis.containers[0], labels=[f'{r * 100:.1f}%\n(n={n:,})' for r, n in zip(stats['churn_rate'], stats['clients'])], padding=3, fontsize=9)
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel('Churn rate, %' if axis is ax[0] else '')
    axis.margins(y=0.2)

fig.suptitle('Churn Rate by Key Drivers', fontsize=14)
plt.tight_layout()
plt.savefig('../images/churn_drivers.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 7))

sns.boxplot(y=df['seat_utilization'][df['is_churned'] == 1], ax=ax[0])
sns.boxplot(y=df['seat_utilization'][df['is_churned'] == 0], ax=ax[1])

ax[0].set_title('Seat utilization among churned customers')
ax[1].set_title('Seat utilization among retained clients')

plt.show()

In [ ]:
df['seat_utilization'][df['is_churned'] == 1].quantile(0.25)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 7))

sns.boxplot(y=df['feature_adoption_rate'][df['is_churned'] == 1], ax=ax[0])
sns.boxplot(y=df['feature_adoption_rate'][df['is_churned'] == 0], ax=ax[1])

ax[0].set_title('Feature Adoption Rate Among Churned Customers')
ax[1].set_title('Feature Adoption Rate Among Retained Clients')

plt.show()

In [ ]:
df['feature_adoption_rate'][df['is_churned'] == 1].quantile(0.25)

In [ ]:
df[['seat_utilization', 'feature_adoption_rate', 'support_tickets_30d']][df['is_churned'] == 1].describe()

In [ ]:
df[['seat_utilization', 'feature_adoption_rate', 'support_tickets_30d']][df['is_churned'] == 0].describe()